# Agentic AI System for Autonomous Test Script Generation — Local Live Demo

**This notebook runs the REAL fine-tuned models.** Unlike the portable demo, every stage here — including Build — executes the actual production code on Apple Silicon (MPS):

- **Build**: the real QLoRA-fine-tuned Phi-3 Mini / Gemma 4 E4B adapter via `agentic_loop.generator.generate`
- **Measure**: the real composite scorer `agentic_loop.scorer.score` (0.40 keyword / 0.30 assertion / 0.30 ROUGE-L)
- **Assess / Decide**: the real BMAD control loop `agentic_loop.loop.run` (threshold 0.60, up to 3 iterations, deterministic feedback)
- **Syntax validation**: real `node --check`
- **Statistics**: the same SciPy routines used for Chapter 5

### Prerequisites (local machine only)
Run with the project `.venv` as the kernel (has `torch`/`transformers`/`peft`), from a machine that has the trained adapters under `fine_tuning/`. Register the kernel once:

```bash
/Users/saif.afzal/Documents/Dissertation/agentic-test-gen/.venv/bin/python -m ipykernel install --user --name diss-venv
```
then select the **diss-venv** kernel.

## 0. Environment setup — load the real modules

In [ ]:
import sys, os
REPO = "/Users/saif.afzal/Documents/Dissertation/agentic-test-gen"
sys.path.insert(0, REPO)
os.chdir(REPO)  # so the adapters' relative paths in MODEL_REGISTRY resolve

import torch
from agentic_loop.generator import generate as real_generate, MODEL_REGISTRY
from agentic_loop.scorer import score as real_score
from agentic_loop.loop import run as bmad_run, DEFAULT_THRESHOLD, DEFAULT_MAX_ITERS

print("MPS available :", torch.backends.mps.is_available())
print("Threshold     :", DEFAULT_THRESHOLD, "| Max iterations:", DEFAULT_MAX_ITERS)
print("Registered models:", list(MODEL_REGISTRY.keys()))

# phi3 = fast (recommended for a live demo); gemma4 = heavier but strongest.
MODEL_KEY = "phi3"

## 1. Sample user stories
Four representative stories spanning categories and complexity tiers. Each carries a Cypress and Playwright reference script (used as the exemplar for the ROUGE-L term).

In [ ]:

stories = [
    {
        "id": "US-101",
        "category": "Authentication",
        "complexity": "simple",
        "text": "As a registered user, I want to log in with my email and password so that I can access my dashboard.",
        "reference": {
            "cypress": '''describe('Login', () => {
  it('logs in with valid credentials', () => {
    cy.visit('/login');
    cy.get('[data-testid="email-input"]').type('user@example.com');
    cy.get('[data-testid="password-input"]').type('Secret123!');
    cy.get('[data-testid="login-button"]').click();
    cy.url().should('include', '/dashboard');
    cy.get('[data-testid="welcome-message"]').should('be.visible');
  });
});''',
            "playwright": '''import { test, expect } from '@playwright/test';

test('logs in with valid credentials', async ({ page }) => {
  await page.goto('/login');
  await page.getByLabel('Email').fill('user@example.com');
  await page.getByLabel('Password').fill('Secret123!');
  await page.getByRole('button', { name: 'Log in' }).click();
  await expect(page).toHaveURL(/dashboard/);
  await expect(page.getByTestId('welcome-message')).toBeVisible();
});''',
        },
    },
    {
        "id": "US-142",
        "category": "CRUD Operations",
        "complexity": "medium",
        "text": "As a project manager, I want to create a new task with a title and due date so that my team knows what to work on next.",
        "reference": {
            "cypress": '''describe('Task creation', () => {
  it('creates a new task with title and due date', () => {
    cy.visit('/tasks');
    cy.get('[data-testid="new-task-button"]').click();
    cy.get('[data-testid="task-title-input"]').type('Prepare release notes');
    cy.get('[data-testid="task-due-date-input"]').type('2026-08-15');
    cy.get('[data-testid="save-task-button"]').click();
    cy.get('[data-testid="task-list"]').should('contain', 'Prepare release notes');
  });
});''',
            "playwright": '''import { test, expect } from '@playwright/test';

test('creates a new task with title and due date', async ({ page }) => {
  await page.goto('/tasks');
  await page.getByRole('button', { name: 'New task' }).click();
  await page.getByLabel('Title').fill('Prepare release notes');
  await page.getByLabel('Due date').fill('2026-08-15');
  await page.getByRole('button', { name: 'Save' }).click();
  await expect(page.getByTestId('task-list')).toContainText('Prepare release notes');
});''',
        },
    },
    {
        "id": "US-207",
        "category": "Form Validation",
        "complexity": "medium",
        "text": "As a new user, I want to see an inline error if I submit the signup form with a mismatched password confirmation so that I can correct it immediately.",
        "reference": {
            "cypress": '''describe('Signup form validation', () => {
  it('shows an error when password confirmation does not match', () => {
    cy.visit('/signup');
    cy.get('[data-testid="password-input"]').type('Secret123!');
    cy.get('[data-testid="confirm-password-input"]').type('Different123!');
    cy.get('[data-testid="signup-button"]').click();
    cy.get('[data-testid="password-error"]').should('be.visible')
      .and('contain', 'Passwords do not match');
  });
});''',
            "playwright": '''import { test, expect } from '@playwright/test';

test('shows an error when password confirmation does not match', async ({ page }) => {
  await page.goto('/signup');
  await page.getByLabel('Password', { exact: true }).fill('Secret123!');
  await page.getByLabel('Confirm password').fill('Different123!');
  await page.getByRole('button', { name: 'Sign up' }).click();
  await expect(page.getByTestId('password-error')).toBeVisible();
  await expect(page.getByTestId('password-error')).toContainText('Passwords do not match');
});''',
        },
    },
    {
        "id": "US-233",
        "category": "Responsive Design",
        "complexity": "complex",
        "text": "As a mobile user, I want to open the navigation via a hamburger menu so that I can reach other pages on a small screen.",
        "reference": {
            "cypress": '''describe('Mobile navigation', () => {
  it('opens the nav menu via the hamburger button on a small viewport', () => {
    cy.viewport('iphone-x');
    cy.visit('/');
    cy.get('[data-testid="hamburger-menu-button"]').click();
    cy.get('[data-testid="mobile-nav"]').should('be.visible');
    cy.get('[data-testid="mobile-nav"]').contains('Products').click();
    cy.url().should('include', '/products');
  });
});''',
            "playwright": '''import { test, expect } from '@playwright/test';

test.use({ viewport: { width: 375, height: 812 } });

test('opens the nav menu via the hamburger button on a small viewport', async ({ page }) => {
  await page.goto('/');
  await page.getByTestId('hamburger-menu-button').click();
  await expect(page.getByTestId('mobile-nav')).toBeVisible();
  await page.getByTestId('mobile-nav').getByText('Products').click();
  await expect(page).toHaveURL(/products/);
});''',
        },
    },
]

import pandas as pd
pd.DataFrame([{"ID": s["id"], "Category": s["category"], "Complexity": s["complexity"], "Story": s["text"]} for s in stories])


## 2. Build stage — the REAL fine-tuned adapter

This calls `agentic_loop.generator.generate`, which loads the QLoRA adapter in-process and generates a candidate script. **The first call loads the model** (~30–60 s for Phi-3 on MPS); subsequent calls are faster. This is the exact code path used to produce the dissertation's results.

In [ ]:
story = stories[0]
framework = "cypress"

script, latency = real_generate(
    model_key=MODEL_KEY, framework=framework,
    user_story=story["text"], category=story["category"],
    complexity=story["complexity"], max_new_tokens=1024,
)
print(f"Generated by {MODEL_KEY}/{framework} in {latency}s:\n")
print(script)

## 3. Measure stage — the REAL composite scorer

`agentic_loop.scorer.score` returns a `QualityScore`: the composite `total`, its three components (`syntax`, `assertion`, `rouge_l`), a `complete` flag (braces balanced?), and the human-readable `feedback` string that the Decide stage sends back to Build on a failure.

In [ ]:
q = real_score(script, framework, exemplar=story["reference"][framework])
print(f"composite total : {q.total:.3f}   (accept threshold {DEFAULT_THRESHOLD})")
print(f"  syntax        : {q.syntax:.3f}")
print(f"  assertion     : {q.assertion:.3f}")
print(f"  ROUGE-L       : {q.rouge_l:.3f}")
print(f"  complete      : {q.complete}")
print(f"  feedback      : {q.feedback}")

## 4. Independent syntax validation — real `node --check`
Decoupled from the composite score, exactly as in the dissertation: a script can pass the BMAD threshold yet still fail to parse (e.g. token truncation). Also reproduces the Section 5.2 Markdown-fence artefact.

In [ ]:

import subprocess, tempfile, os

def check_syntax(candidate, framework):
    ext = ".cy.js" if framework == "cypress" else ".spec.ts"
    # node --check parses JS; for a live, dependency-free check we validate the
    # .ts sample as plain JS syntax (import/await/async are valid ES module
    # syntax under --input-type=module), matching the dissertation's actual
    # approach of parsing with Node's own engine rather than a framework-specific
    # compiler.
    with tempfile.NamedTemporaryFile(suffix=".mjs", mode="w", delete=False) as f:
        f.write(candidate)
        path = f.name
    try:
        result = subprocess.run(
            ["node", "--check", path],
            capture_output=True, text=True, timeout=10,
        )
        return result.returncode == 0, result.stderr.strip()
    finally:
        os.unlink(path)

ok, err = check_syntax(stories[0]["reference"]["cypress"], "cypress")
print("Reference script valid:", ok, err or "(no errors)")

# Demonstrate the exact defect the dissertation reports catching (Section 5.2):
# an unstripped Markdown code fence corrupting the syntax check. This needs a
# script that uses a backtick template literal for a parameterised selector
# (a realistic pattern -- e.g. selecting a row by id) for the fence's stray
# backticks to actually collide with the code's own backticks the way the
# dissertation describes, rather than merely wrapping the code in an inert
# (still-parseable) string.
templated_script = '''import { test, expect } from '@playwright/test';

test(`selects the row for id ${1}`, async ({ page }) => {
  await page.goto('/rows/1');
  await expect(page.getByTestId(`row-1`)).toBeVisible();
});'''

ok_clean, err_clean = check_syntax(templated_script, "playwright")
print("\nUn-fenced templated script valid:", ok_clean, err_clean or "(no errors)")

fenced = "```typescript\n" + templated_script + "\n```"
ok_fenced, err_fenced = check_syntax(fenced, "playwright")
print("\nSame script wrapped in an un-stripped Markdown fence:")
print("Valid:", ok_fenced)
print("Error:", err_fenced[:300])


## 5. The BMAD loop — the REAL Build→Measure→Assess→Decide

`agentic_loop.loop.run` is the actual production loop: it generates with the fine-tuned adapter, scores, checks the 0.60 threshold, and on a shortfall constructs deterministic corrective feedback and regenerates (up to 3 attempts, keeping the best).

**Safe default for a live demo:** the cell below runs a **single story on a single framework** — one model resident (~8 GB), finishes in well under a minute on Phi-3. The full 4-stories × 2-frameworks sweep is kept below as a commented block; uncomment it only when the machine is idle (it loads a second model, peaking ~16–20 GB, and takes ~8–15 min on Phi-3).

In [ ]:
import pandas as pd, time

# ---- SAFE DEFAULT: one story, one framework (fast, ~8 GB, < 1 min on phi3) ----
DEMO_STORY = stories[0]          # pick any: stories[0..3]
DEMO_FRAMEWORK = "cypress"       # "cypress" or "playwright"

results = []
r = bmad_run(
    tc_id=DEMO_STORY["id"], user_story=DEMO_STORY["text"],
    framework=DEMO_FRAMEWORK, model_key=MODEL_KEY,
    category=DEMO_STORY["category"], complexity=DEMO_STORY["complexity"],
    exemplar=DEMO_STORY["reference"][DEMO_FRAMEWORK],
)
results.append(r)
print(f"  {r.tc_id}/{r.framework:<10} accepted={r.accepted!s:<5} "
      f"score={r.best_score:.3f}  iters={r.iterations}  ({r.total_latency_s:.1f}s)")

# ---- FULL SWEEP (idle machine only): 4 stories x 2 frameworks -----------------
# WARNING: loads a second model (peak ~16-20 GB on phi3, more on gemma4) and
# runs up to ~20 live generations (~8-15 min on phi3). Uncomment to run.
#
# results = []
# for s in stories:
#     for fw in ["cypress", "playwright"]:
#         r = bmad_run(
#             tc_id=s["id"], user_story=s["text"], framework=fw, model_key=MODEL_KEY,
#             category=s["category"], complexity=s["complexity"],
#             exemplar=s["reference"][fw],
#         )
#         results.append(r)
#         print(f"  {s['id']}/{fw:<10} accepted={r.accepted!s:<5} "
#               f"score={r.best_score:.3f}  iters={r.iterations}  ({r.total_latency_s:.1f}s)")


### Summary table (live results)

In [ ]:
rows = []
for r in results:
    ok, _ = check_syntax(r.final_script, r.framework)
    rows.append({"Story": r.tc_id, "Framework": r.framework,
                 "Accepted": r.accepted, "Iterations": r.iterations,
                 "Best composite": round(r.best_score, 3),
                 "Syntax valid (node --check)": ok})
summary = pd.DataFrame(rows)
summary

**Worth pointing out live:** the *Accepted* column reflects the composite-score gate, while *Syntax valid* is the independent `node --check`. A row can be accepted yet fail the external check — the exact decoupling the dissertation argues for, and why syntax validity is reported separately in Chapter 5.

## 6. Reproducing the Chapter 5 statistical methodology

The live loop above runs a handful of stories for demonstration. The published statistics come from the full 279×8 evaluation. Below, the per-story score vectors are reconstructed to the **real** Table 5.3 means/SDs (preserving the *paired* design), then the **same SciPy routines used in Chapter 5** are run on them — printing the computed statistic next to the dissertation's reported value.

In [ ]:

import numpy as np
from scipy import stats

rng = np.random.default_rng(42)
N = 279

# Published mean / std from Table 5.3 (real, reported numbers)
table_5_3 = {
    ("Gemma 4 E4B", "Playwright"): (0.487, 0.076),
    ("Gemma 4 E4B", "Cypress"):    (0.441, 0.107),
    ("Phi-3 Mini",  "Playwright"): (0.432, 0.067),
    ("Phi-3 Mini",  "Cypress"):    (0.383, 0.086),
    ("Claude Haiku","Playwright"): (0.341, 0.065),
    ("GPT-4o-mini", "Playwright"): (0.339, 0.067),
    ("GPT-4o-mini", "Cypress"):    (0.304, 0.071),
    ("Claude Haiku","Cypress"):    (0.294, 0.075),
}

# Shared per-story "difficulty" term preserves the *paired* design (the same
# 279 stories are scored by every system) rather than generating 8 independent
# unpaired samples.
difficulty = rng.normal(0, 0.02, size=N)

synthetic = {}
for (model, framework), (mean, std) in table_5_3.items():
    noise = rng.normal(0, std, size=N)
    scores = np.clip(mean + difficulty + noise, 0, 1)
    synthetic[(model, framework)] = scores

print("Reconstructed sample means (should match Table 5.3 closely):")
for k, v in synthetic.items():
    print(f"  {k[0]:<14} {k[1]:<11} mean={v.mean():.3f}  std={v.std():.3f}")


In [ ]:

# --- One-way ANOVA, all eight systems (cf. Table 5.4) ---
groups = list(synthetic.values())
f_stat, p_val = stats.f_oneway(*groups)
print(f"ANOVA, all 8 systems:        F={f_stat:.2f}, p={p_val:.3e}   (dissertation: F=225.63, p=1.01e-253)")

cypress_groups = [v for k, v in synthetic.items() if k[1] == "Cypress"]
f_c, p_c = stats.f_oneway(*cypress_groups)
print(f"ANOVA, Cypress only:         F={f_c:.2f}, p={p_c:.3e}   (dissertation: F=182.90, p=2.18e-96)")

playwright_groups = [v for k, v in synthetic.items() if k[1] == "Playwright"]
f_p, p_p = stats.f_oneway(*playwright_groups)
print(f"ANOVA, Playwright only:      F={f_p:.2f}, p={p_p:.3e}   (dissertation: F=311.72, p=7.74e-147)")


In [ ]:

# --- Paired Wilcoxon signed-rank tests (cf. Table 5.5) ---
pairs = [
    ("Gemma 4 E4B", "Claude Haiku", "Cypress"),
    ("Gemma 4 E4B", "Claude Haiku", "Playwright"),
    ("Gemma 4 E4B", "GPT-4o-mini", "Cypress"),
    ("Gemma 4 E4B", "GPT-4o-mini", "Playwright"),
    ("Phi-3 Mini", "Claude Haiku", "Cypress"),
    ("Phi-3 Mini", "Claude Haiku", "Playwright"),
    ("Phi-3 Mini", "GPT-4o-mini", "Cypress"),
    ("Phi-3 Mini", "GPT-4o-mini", "Playwright"),
    ("Gemma 4 E4B", "Phi-3 Mini", "Cypress"),
    ("Gemma 4 E4B", "Phi-3 Mini", "Playwright"),
]

rows = []
for a, b, fw in pairs:
    x, y = synthetic[(a, fw)], synthetic[(b, fw)]
    stat, p = stats.wilcoxon(x, y)
    rows.append({"Comparison": f"{a} vs. {b}", "Framework": fw,
                  "Mean diff.": round(float(x.mean() - y.mean()), 3),
                  "p-value": f"{p:.2e}", "Significant (p<0.05)": p < 0.05})

pd.DataFrame(rows)


In [ ]:

# --- Chi-square test of independence: syntax validity by system category ---
# These are the REAL, reported figures from Section 5.6 (Table 5.6 aggregate),
# not synthetic: 1,110 valid / 6 invalid on each side of an identical 2,232-script
# full evaluation.
contingency = np.array([
    [1110, 6],   # baseline: valid, invalid
    [1110, 6],   # fine-tuned: valid, invalid
])
chi2, p, dof, expected = stats.chi2_contingency(contingency)
print(f"Chi-square test of independence (real data): chi2={chi2:.2f}, df={dof}, p={p:.2f}")
print("(dissertation reports: chi2 = 0.00, df = 1, p = 1.00 -- matches exactly)")


## 7. Visualising the real, published results (Tables 5.2 & 5.3)
These plot the dissertation's actual reported numbers directly — no synthetic data.

In [ ]:

import matplotlib.pyplot as plt

rouge_data = [
    ("GPT-4o-mini\n(Cypress)", 0.304), ("Claude Haiku\n(Cypress)", 0.294),
    ("Phi-3 Mini\n(Cypress)", 0.383), ("Gemma 4 E4B\n(Cypress)", 0.441),
    ("GPT-4o-mini\n(Playwright)", 0.339), ("Claude Haiku\n(Playwright)", 0.341),
    ("Phi-3 Mini\n(Playwright)", 0.432), ("Gemma 4 E4B\n(Playwright)", 0.487),
]
labels = [d[0] for d in rouge_data]
values = [d[1] for d in rouge_data]
colors = ["#9fb8d6"] * 2 + ["#4a7fb5"] * 2 + ["#9fb8d6"] * 2 + ["#4a7fb5"] * 2

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].bar(labels, values, color=colors)
axes[0].set_ylabel("Mean ROUGE-L F1")
axes[0].set_title("Table 5.3 — Semantic quality, all 8 systems (n=279 each)")
axes[0].tick_params(axis='x', rotation=45)
axes[0].set_ylim(0, 0.55)

accept_data = [
    ("Phi-3 Mini\n(Cypress)", 97.5), ("Phi-3 Mini\n(Playwright)", 90.0),
    ("Gemma 4 E4B\n(Cypress)", 100.0), ("Gemma 4 E4B\n(Playwright)", 98.6),
]
axes[1].bar([d[0] for d in accept_data], [d[1] for d in accept_data], color="#4a7fb5")
axes[1].set_ylabel("BMAD Accept Rate (%)")
axes[1].set_title("Table 5.2 — Agentic loop outcomes, complete dataset")
axes[1].set_ylim(0, 105)
for i, d in enumerate(accept_data):
    axes[1].text(i, d[1] + 1.5, f"{d[1]}%", ha="center")

plt.tight_layout()
plt.savefig("results_summary.png", dpi=130)
plt.show()


## 8. Talking points for the defense (local run)

- **Every stage here is the production code, running live on this machine** — Build calls the real QLoRA-fine-tuned adapter via `agentic_loop.generator`, Measure/Assess/Decide are `agentic_loop.scorer` and `agentic_loop.loop`, syntax validation is real `node --check`, and the statistics are the same SciPy functions used for Chapter 5. Nothing above is a stand-in.
- **The only thing not reproduced live is the full 279×8 batch**, which takes hours; it is summarised from the committed `results/` and the Chapter 5 tables. The live loop demonstrates the identical mechanism on a representative sample.
- **If a story is accepted on the first attempt**, that is the real behaviour — 82% of records (914/1,116) passed on attempt 1 in the full run (average 1.23 iterations). To show a correction happening live, pick a harder story (a complex-tier or Playwright one), where a second attempt is more likely.
- **Model choice:** `MODEL_KEY='phi3'` is used for responsiveness; switch to `'gemma4'` to demonstrate the strongest configuration (heavier load, slower per call).
- **The correction signal is deterministic, not a second model:** the feedback returned by the scorer is rule-derived (missing keywords, unbalanced braces, low assertion count), so the loop is reproducible and adds no extra inference cost — consistent with the small-model efficiency thesis.